# S49_02 — LoRA and QLoRA for LLMs

LoRA was introduced in S45_04 for BERT-scale models. This notebook focuses on its application to large decoder-only LLMs (Llama, Mistral, Qwen) — the main use case in practice.

## QLoRA: 4-bit quantization + LoRA

**QLoRA** (Dettmers et al. 2023) extends LoRA by quantizing the frozen base model to 4-bit precision using NF4 (NormalFloat4) — a quantization format optimized for normally-distributed neural network weights. This reduces a 7B model from ~14GB (fp16) to ~5GB (4-bit), fitting on a consumer GPU.

```
Memory per 7B model:
  FP32: 28 GB
  FP16/BF16: 14 GB
  INT8: 7 GB
  NF4 (QLoRA): 4–5 GB  ← fits on 8GB GPU
```

In [ ]:
# pip install transformers peft bitsandbytes accelerate
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',         # NormalFloat4 — best for LLM weights
    bnb_4bit_compute_dtype=torch.bfloat16,  # compute in bf16 (not fp32)
    bnb_4bit_use_double_quant=True,     # double quantization saves ~0.4 bits/param more
)

# Load quantized model (requires a CUDA GPU with bitsandbytes installed)
model_name = 'meta-llama/Llama-3.2-1B'  # smallest Llama for demo

# Uncomment to actually load:
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map='auto',      # distribute across available GPUs/CPU
#     torch_dtype=torch.bfloat16,
# )
# tokenizer = AutoTokenizer.from_pretrained(model_name)

print('4-bit QLoRA config defined')
print('Requires: CUDA GPU + bitsandbytes + sufficient VRAM')

In [ ]:
# LoRA config for decoder-only LLMs
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                        # rank — 8 or 16 is typical
    lora_alpha=32,               # scaling = alpha/r; typically 2×r
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',  # attention projections
        'gate_proj', 'up_proj', 'down_proj',       # MLP layers (often helps)
    ],
    lora_dropout=0.05,
    bias='none',
)

# After loading model:
# model.enable_input_require_grads()   # needed for gradient checkpointing with PEFT
# peft_model = get_peft_model(model, lora_config)
# peft_model.print_trainable_parameters()
# → trainable params: ~8M / 1.2B (0.7%)

print('QLoRA parameter estimate for Llama-3.2-1B:')
total = 1_235_814_400  # 1.2B params
# r=16, 32 layers, 4 attention + 3 MLP = 7 matrices per layer, 2 per LoRA
lora_params = 16 * 32 * 7 * 2 * 2048  # rough estimate
print(f'  Total params: {total:,}')
print(f'  LoRA trainable: ~{lora_params:,} ({lora_params/total*100:.2f}%)')
print(f'  4-bit base model memory: ~{total * 0.5 / 1e9:.1f} GB (4 bits = 0.5 bytes/param)')

## Data format for SFT

In [ ]:
# Standard chat format (matches Llama-3, Mistral instruction template)

def format_instruction_llama3(system, user, assistant):
    """Llama-3 instruction format."""
    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system}<|eot_id|><|start_header_id|>user<|end_header_id|>
{user}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
{assistant}<|eot_id|>"""

# Example training sample
sample = format_instruction_llama3(
    system='You are a helpful data science tutor.',
    user='What is the difference between bagging and boosting?',
    assistant='Bagging trains models in parallel on bootstrap samples and averages predictions (reduces variance). Boosting trains models sequentially, each correcting the errors of the previous (reduces bias). Random Forest = bagging; XGBoost/AdaBoost = boosting.',
)
print(sample)
print(f'\nToken count (approx): {len(sample.split()) * 1.3:.0f}')

In [ ]:
# Create a tiny dataset in the JSONL format used by most training frameworks
import json

training_data = [
    {
        'instruction': 'What is gradient descent?',
        'input': '',
        'output': 'Gradient descent is an optimization algorithm that iteratively updates model parameters by moving in the direction opposite to the gradient of the loss function, scaled by a learning rate.',
    },
    {
        'instruction': 'Explain overfitting in one sentence.',
        'input': '',
        'output': 'Overfitting occurs when a model learns the training data too well, including its noise, causing poor performance on unseen data.',
    },
    {
        'instruction': 'What library should I use for gradient boosting in 2026?',
        'input': '',
        'output': 'For tabular data, LightGBM is usually the fastest choice; XGBoost and CatBoost are also excellent. For very large datasets, consider GPU-accelerated versions.',
    },
]

# Write to JSONL
# with open('training_data.jsonl', 'w') as f:
#     for sample in training_data:
#         f.write(json.dumps(sample) + '\n')

print(f'{len(training_data)} training examples defined')
print('Note: real fine-tunes need 100-10k+ examples')

## Training with TRL SFTTrainer

In [ ]:
# TRL (Transformer Reinforcement Learning) library provides SFTTrainer
# pip install trl

# Full QLoRA fine-tuning script (requires GPU)
qlora_script = '''
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from datasets import load_dataset
import torch

model_name = "meta-llama/Llama-3.2-1B"

# Load 4-bit
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    ),
    device_map="auto",
)
model.enable_input_require_grads()
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# LoRA
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj","v_proj"], task_type="CAUSAL_LM")

# Train
trainer = SFTTrainer(
    model=model,
    args=TrainingArguments(output_dir="./qlora-out", num_train_epochs=3, per_device_train_batch_size=4,
                           gradient_accumulation_steps=4, learning_rate=2e-4, bf16=True),
    train_dataset=load_dataset("json", data_files="training_data.jsonl", split="train"),
    peft_config=lora_config,
    tokenizer=tokenizer,
    dataset_text_field="output",
)
trainer.train()
trainer.save_model("./qlora-out")
'''

print('QLoRA fine-tuning script (requires GPU):')
print(qlora_script[:500] + '...')
print('\nSee S49_05_finetuning_with_unsloth.ipynb for the Unsloth version (2–3× faster)')

Next: [S49_03_instruction_tuning.ipynb](./S49_03_instruction_tuning.ipynb)